# Synapse T4 GPU Benchmark

Connects 2 headless Chrome nodes (with WebGPU on T4) to the Synapse coordinator,
then runs inference benchmarks to measure tok/sec.

**Requirements:** Colab T4 GPU runtime

In [ ]:
# Cell 1: Verify GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
# Cell 2: Install headless Chrome + dependencies
%%bash
set -e

# Install Chrome
if ! command -v google-chrome-stable &> /dev/null; then
  echo "Installing Chrome..."
  apt-get update -qq
  apt-get install -y -qq wget gnupg2 > /dev/null 2>&1
  wget -q -O - https://dl.google.com/linux/linux_signing_key.pub | apt-key add - 2>/dev/null
  echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' > /etc/apt/sources.list.d/google-chrome.list
  apt-get update -qq
  apt-get install -y -qq google-chrome-stable > /dev/null 2>&1
  echo "Chrome installed: $(google-chrome-stable --version)"
else
  echo "Chrome already installed: $(google-chrome-stable --version)"
fi

# Install Node.js 20
if ! command -v node &> /dev/null || [[ $(node -v) != v20* ]]; then
  echo "Installing Node.js 20..."
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
  apt-get install -y -qq nodejs > /dev/null 2>&1
  echo "Node.js installed: $(node -v)"
else
  echo "Node.js already installed: $(node -v)"
fi

# Install puppeteer-core locally
mkdir -p /tmp/synapse-work
cd /tmp/synapse-work
npm init -y > /dev/null 2>&1
npm install puppeteer-core > /dev/null 2>&1
echo "puppeteer-core installed at /tmp/synapse-work/node_modules"
echo "Setup complete."

In [ ]:
# Cell 3: Test Chrome WebGPU on T4
%%bash
cat > /tmp/synapse-work/test_webgpu.mjs << 'SCRIPT'
import puppeteer from 'puppeteer-core';

const browser = await puppeteer.launch({
  executablePath: '/usr/bin/google-chrome-stable',
  headless: 'new',
  args: [
    '--no-sandbox',
    '--use-angle=vulkan',
    '--enable-features=Vulkan',
    '--disable-vulkan-surface',
    '--enable-unsafe-webgpu',
  ],
});

const page = await browser.newPage();

// Check WebGPU
const result = await page.evaluate(async () => {
  if (!navigator.gpu) return { ok: false, reason: 'navigator.gpu not available' };
  try {
    const adapter = await navigator.gpu.requestAdapter();
    if (!adapter) return { ok: false, reason: 'no adapter returned' };
    const info = await adapter.requestAdapterInfo();
    const device = await adapter.requestDevice();
    return {
      ok: true,
      vendor: info.vendor,
      architecture: info.architecture,
      description: info.description,
      device: info.device,
      maxBufferSize: adapter.limits.maxBufferSize,
      maxComputeWGSize: adapter.limits.maxComputeWorkgroupSizeX,
    };
  } catch (e) {
    return { ok: false, reason: e.message };
  }
});

console.log(JSON.stringify(result, null, 2));
await browser.close();
SCRIPT

cd /tmp/synapse-work && node test_webgpu.mjs

In [ ]:
#@title Cell 4: Set coordinator URL
COORDINATOR_URL = "http://136.112.71.75:8080" #@param {type:"string"}

In [ ]:
# Cell 5: Launch 2 headless compute nodes and connect to coordinator
import subprocess, time, threading, sys, os

node_script = r'''
import puppeteer from 'puppeteer-core';

const COORDINATOR = process.env.COORDINATOR_URL;
const NUM_NODES = 2;

const CHROME_FLAGS = [
  '--no-sandbox',
  '--headless=new',
  '--use-angle=vulkan',
  '--enable-features=Vulkan',
  '--disable-vulkan-surface',
  '--enable-unsafe-webgpu',
  '--no-first-run',
  '--no-default-browser-check',
  '--disable-background-timer-throttling',
  '--disable-renderer-backgrounding',
  '--disable-backgrounding-occluded-windows',
];

console.log(`Coordinator: ${COORDINATOR}`);
console.log(`Launching ${NUM_NODES} headless nodes...\n`);

const browser = await puppeteer.launch({
  executablePath: '/usr/bin/google-chrome-stable',
  headless: 'new',
  args: CHROME_FLAGS,
});

// Check GPU first
const gpuPage = await browser.newPage();
await gpuPage.goto('chrome://gpu', { waitUntil: 'networkidle2' });
const gpuInfo = await gpuPage.evaluate(() => {
  const text = document.body.innerText;
  return text.split('\n').filter(l =>
    l.includes('WebGPU') || l.includes('Vulkan') || l.includes('GL_RENDERER')
  ).join('\n');
});
console.log('GPU Status:', gpuInfo);
await gpuPage.close();

const pages = [];
for (let i = 0; i < NUM_NODES; i++) {
  const page = await browser.newPage();

  page.on('console', msg => {
    const text = msg.text();
    if (text.includes('[Synapse]') || text.includes('WebGPU') ||
        text.includes('shard') || text.includes('connected') ||
        text.includes('ready') || text.includes('assigned') ||
        text.includes('tensor') || text.includes('loaded') ||
        text.includes('error') || text.includes('Error')) {
      console.log(`[Node ${i}] ${text}`);
    }
  });

  page.on('pageerror', err => console.error(`[Node ${i}] ERROR: ${err.message}`));

  const url = `${COORDINATOR}/node/index.html`;
  console.log(`[Node ${i}] Opening ${url}`);
  await page.goto(url, { waitUntil: 'networkidle2', timeout: 60000 });

  // Check WebGPU
  const gpu = await page.evaluate(async () => {
    if (!navigator.gpu) return { ok: false, reason: 'no navigator.gpu' };
    const adapter = await navigator.gpu.requestAdapter();
    if (!adapter) return { ok: false, reason: 'no adapter' };
    const info = await adapter.requestAdapterInfo();
    return { ok: true, desc: info.description || info.device || 'T4' };
  });

  console.log(`[Node ${i}] WebGPU: ${gpu.ok ? gpu.desc : 'FAILED - ' + gpu.reason}`);
  pages.push(page);

  if (i < NUM_NODES - 1) await new Promise(r => setTimeout(r, 3000));
}

console.log(`\n${pages.length}/${NUM_NODES} nodes launched.`);
console.log('Waiting for shard loading (120s timeout)...\n');

// Wait for nodes to become ready
for (let i = 0; i < pages.length; i++) {
  try {
    await pages[i].waitForFunction(() => {
      const dot = document.getElementById('statusDot');
      return dot && dot.classList.contains('ready');
    }, { timeout: 120000 });
    console.log(`[Node ${i}] READY`);
  } catch (e) {
    const status = await pages[i].evaluate(() => {
      return document.getElementById('statusText')?.textContent || 'unknown';
    });
    console.log(`[Node ${i}] Not ready after 120s. Status: ${status}`);
  }
}

// Print final status
for (let i = 0; i < pages.length; i++) {
  const info = await pages[i].evaluate(() => ({
    status: document.getElementById('statusText')?.textContent,
    shard: document.getElementById('shardInfo')?.textContent,
    layers: document.getElementById('layerInfo')?.textContent,
    gpu: document.getElementById('gpuVendor')?.textContent,
    tensors: document.getElementById('tensorCount')?.textContent,
  }));
  console.log(`[Node ${i}] ${JSON.stringify(info)}`);
}

console.log('\nNODES_READY');
console.log('Keeping nodes alive for benchmark. Run Cell 6 to benchmark.');

// Keep alive — will be killed when cell is interrupted
await new Promise(() => {});
'''

with open('/tmp/synapse-work/launch_nodes.mjs', 'w') as f:
    f.write(node_script)

os.environ['COORDINATOR_URL'] = COORDINATOR_URL

# Run from /tmp/synapse-work where puppeteer-core is installed
proc = subprocess.Popen(
    ['node', 'launch_nodes.mjs'],
    cwd='/tmp/synapse-work',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env={**os.environ},
    text=True,
    bufsize=1,
)

# Stream output until nodes are ready or timeout
ready = False
start = time.time()
while time.time() - start < 180:  # 3 min timeout
    line = proc.stdout.readline()
    if not line:
        break
    print(line.rstrip())
    sys.stdout.flush()
    if 'NODES_READY' in line:
        ready = True
        break

if ready:
    print('\n=== Nodes connected and ready! Run Cell 6 for benchmark ===')
else:
    print('\n=== Timeout or error. Check output above. ===')

In [ ]:
# Cell 6: Run inference benchmark (tok/sec)
!pip install -q websockets

import asyncio
import json
import time
import websockets

PROMPTS = [
    "The meaning of life is",
    "In a galaxy far far away",
    "def fibonacci(n):",
    "The quick brown fox",
    "Scientists recently discovered that",
    "Once upon a time there was",
    "The capital of France is",
    "import numpy as np\n",
]

MAX_TOKENS = 20

async def benchmark():
    ws_url = COORDINATOR_URL.replace('http://', 'ws://').replace('https://', 'wss://') + '?type=prompt'
    print(f'Connecting to {ws_url}...')

    async with websockets.connect(ws_url, ping_interval=30) as ws:
        # Wait for pipeline ready
        print('Waiting for pipeline...')
        while True:
            msg = json.loads(await asyncio.wait_for(ws.recv(), timeout=120))
            if msg.get('type') == 'PIPELINE_READY':
                nodes = msg.get('pipeline', [])
                print(f'Pipeline ready: {len(nodes)} nodes')
                break
            elif msg.get('type') == 'TOPOLOGY_UPDATE':
                nodes = msg.get('nodes', [])
                ready = sum(1 for n in nodes if n.get('status') == 'ready')
                print(f'  Topology: {ready}/{len(nodes)} nodes ready')

        results = []
        total_tokens = 0
        total_time = 0

        for i, prompt in enumerate(PROMPTS):
            # Tokenize via coordinator (send raw text)
            req = {
                'type': 'PROMPT_SUBMIT',
                'prompt': prompt,
                'maxTokens': MAX_TOKENS,
            }
            await ws.send(json.dumps(req))

            tokens = []
            t0 = time.time()
            first_token_time = None
            error = None

            while True:
                raw = await asyncio.wait_for(ws.recv(), timeout=60)
                msg = json.loads(raw)

                if msg['type'] == 'TOKEN_GENERATED':
                    if first_token_time is None:
                        first_token_time = time.time()
                    tokens.append(msg.get('text', msg.get('token', '')))

                elif msg['type'] == 'GENERATION_DONE':
                    elapsed = time.time() - t0
                    ttft = (first_token_time - t0) if first_token_time else elapsed
                    n_tok = len(tokens)
                    tok_sec = n_tok / elapsed if elapsed > 0 else 0
                    # Decode throughput excludes prefill
                    decode_time = elapsed - ttft if first_token_time else elapsed
                    decode_tps = (n_tok - 1) / decode_time if decode_time > 0 and n_tok > 1 else 0

                    server_tps = msg.get('tokensPerSecond', 0)

                    print(f'  [{i+1}/{len(PROMPTS)}] "{prompt[:30]}..."')
                    print(f'    Tokens: {n_tok} | Total: {elapsed:.2f}s | TTFT: {ttft*1000:.0f}ms')
                    print(f'    E2E: {tok_sec:.1f} tok/s | Decode: {decode_tps:.1f} tok/s | Server: {server_tps} tok/s')
                    print(f'    Output: {" ".join(str(t) for t in tokens[:10])}...')

                    results.append({
                        'prompt': prompt,
                        'tokens': n_tok,
                        'elapsed_s': elapsed,
                        'ttft_ms': ttft * 1000,
                        'e2e_tok_sec': tok_sec,
                        'decode_tok_sec': decode_tps,
                        'server_tok_sec': server_tps,
                    })
                    total_tokens += n_tok
                    total_time += elapsed
                    break

                elif msg['type'] == 'INFER_ERROR':
                    error = msg.get('error', 'unknown')
                    print(f'  [{i+1}] ERROR: {error}')
                    break

        # Summary
        print('\n' + '=' * 60)
        print('BENCHMARK RESULTS')
        print('=' * 60)
        if results:
            avg_e2e = sum(r['e2e_tok_sec'] for r in results) / len(results)
            avg_decode = sum(r['decode_tok_sec'] for r in results) / len(results)
            avg_ttft = sum(r['ttft_ms'] for r in results) / len(results)
            avg_server = sum(r['server_tok_sec'] for r in results) / len(results)
            overall = total_tokens / total_time if total_time > 0 else 0

            print(f'Prompts:          {len(results)}')
            print(f'Total tokens:     {total_tokens}')
            print(f'Total time:       {total_time:.2f}s')
            print(f'Overall:          {overall:.1f} tok/s')
            print(f'Avg E2E:          {avg_e2e:.1f} tok/s')
            print(f'Avg Decode:       {avg_decode:.1f} tok/s')
            print(f'Avg TTFT:         {avg_ttft:.0f} ms')
            print(f'Avg Server:       {avg_server:.1f} tok/s')
            print()

            # Compare against targets
            targets = [
                ('Baseline',          16),
                ('KV cache',          48),
                ('Binary+int8',       80),
                ('Peer-to-peer',     140),
                ('Prediction',       350),
                ('Early exit',       450),
                ('Speculative',      800),
                ('Head pruning',     900),
                ('Theoretical max', 1000),
            ]
            print('TARGET COMPARISON:')
            for name, target in targets:
                hit = '>>>' if avg_decode >= target else '   '
                bar_len = int(min(avg_decode / target, 1.0) * 20)
                bar = '#' * bar_len + '.' * (20 - bar_len)
                pct = avg_decode / target * 100
                print(f'  {hit} {name:20s} {target:6d} tok/s [{bar}] {pct:5.1f}%')
        else:
            print('No successful results.')

await benchmark()

In [ ]:
# Cell 7: Cleanup — stop nodes
try:
    proc.terminate()
    proc.wait(timeout=5)
    print('Nodes stopped.')
except:
    proc.kill()
    print('Nodes killed.')